In [1]:
import torch
from torch import nn
import numpy as np
from string import Template
import math

import os 
import sys
sys.path.append(os.path.abspath("./"))
from input_image_gen import generate_input_file
from conv_layer_generator import *
from dense_layer_generator import *
from codebooks_defs_generator import *

In [2]:
def conv_out_size(in_size, stride, padding, k_size):
    return int(((in_size - k_size + (2 * padding)) / stride) + 1) 

def max_pool_out_size(in_size, pool_size, stride):
    return int(((in_size - pool_size) / (stride)) + 1)

In [3]:

TILE_L2_SIZE = 6
TILE_L1_SIZE = 12

CODEBOOK_SIZE = 4
SVE_LANES = 4

N_LEARNERS = 4

In [4]:
# Parameters
TILE_L2_SIZE = 6
TILE_L1_SIZE = 12
CODEBOOK_SIZE = 16
SVE_LANES = 64
N_LEARNERS = 8


In [5]:

USE_F16 = False

SAME_SEQ = False

USE_BIAS = True

USE_CODEBOOKS = True




IN_CHANNELS = 3
IN_HEIGHT = 32
IN_WIDTH = IN_HEIGHT


####################
## Chris' AlexNet ##
####################

conv_0 = {"type": "conv",
          "in_ch": IN_CHANNELS, 
          "out_ch": 64,
          "k_size": 5, 
          "stride": 1, 
          "padding": 2}

max_pool_0 = {"type": "maxpool",
              "size": 3,
              "stride": 2}

conv_1 = {"type": "conv",
          "in_ch": conv_0["out_ch"], 
          "out_ch": 192,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

max_pool_1 = {"type": "maxpool",
              "size": 3,
              "stride":2}

conv_2 = {"type": "conv",
          "in_ch": conv_1["out_ch"], 
          "out_ch": 384,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_3 = {"type": "conv",
          "in_ch": conv_2["out_ch"], 
          "out_ch": 256,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

conv_4 = {"type": "conv",
          "in_ch": conv_3["out_ch"], 
          "out_ch": 256,
          "k_size": 3,
          "stride": 1,
          "padding": 1}

glob_avg_pool = {"type": "glob_avg_pool"}

dense_5 = {"type": "dense",
           "out_size": 4096}

dense_6 = {"type": "dense",
           "out_size": 4096}

dense_7 = {"type": "dense",
           "out_size": 10}

NN_structure = [conv_0, max_pool_0, conv_1, max_pool_1, conv_2, conv_3, conv_4, glob_avg_pool, dense_5, dense_6, dense_7]

In [6]:
# OUT_FOLDER = "./generated_headers/"
OUT_FOLDER = "./../alexnet_definitions/"

generate_cb_definitions(OUT_FOLDER + "codebooks_def.h", N_LEARNERS, CODEBOOK_SIZE, SVE_LANES, USE_BIAS, USE_F16, SAME_SEQ, USE_CODEBOOKS)
input_values = generate_input_file(OUT_FOLDER + "input_image.h", IN_CHANNELS, IN_HEIGHT, IN_WIDTH, USE_F16)

# These are in case the layer is a conv or max pool
input_ch = IN_CHANNELS
input_height = IN_HEIGHT
input_width = IN_WIDTH

# This is in case the layer is a dense layer
in_size = IN_CHANNELS * IN_HEIGHT * IN_WIDTH


# This list is to save all the kernel values, so to test with torch
kernels = []

# This list is to save all the dense layer values, so   to test with torch
dense_weights = []

# Final torch network, one per learner
network = [nn.ModuleDict({}) for _ in range(N_LEARNERS)]

for lay_cnt, layer in enumerate(NN_structure):
    print("[{}] {}".format(lay_cnt, layer["type"]))
    print("\t", layer)

    if layer['type'] == "conv":
        in_shape = (layer["in_ch"], input_height, input_width)
        out_channels = layer["out_ch"]
        out_height = conv_out_size(input_height, layer["stride"], layer["padding"], layer["k_size"])
        out_width = conv_out_size(input_width, layer["stride"], layer["padding"], layer["k_size"])
        out_shape = (out_channels, out_height, out_width)

        # Get the codebooks values for the learners and generate the header
        cb_values, k_values, biases_values = generate_kernel_header_file(SAME_SEQ, OUT_FOLDER + "conv_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, out_channels, layer["in_ch"], layer["k_size"], layer['stride'], layer["padding"], TILE_L2_SIZE, TILE_L1_SIZE, USE_F16, USE_CODEBOOKS)
        kernels.append(k_values)


    
        # Per each learner network, force the kernel values and append the layer to the learner network
        for learner in range(N_LEARNERS):

            new_conv = nn.Conv2d(in_channels=layer['in_ch'],
                                                        out_channels=layer["out_ch"],
                                                        kernel_size=(layer["k_size"], layer["k_size"]),
                                                        stride = layer['stride'],
                                                        padding=layer["padding"],
                                                        bias = USE_BIAS)
            with torch.no_grad():
                new_conv.weight.copy_(torch.tensor(k_values[learner]).view(layer["out_ch"], layer['in_ch'], layer["k_size"], layer["k_size"]))

                if USE_BIAS:
                    new_conv.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["conv{}".format(lay_cnt)] = new_conv


        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width


    elif layer["type"] == "maxpool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = max_pool_out_size(input_height, layer["size"], layer["stride"])
        out_width = max_pool_out_size(input_width, layer["size"], layer["stride"])
        out_shape = (out_channels, out_height, out_width)

        # Per each learner network, append the layer to the network
        for learner in range(N_LEARNERS):
            new_maxpool = nn.MaxPool2d(layer["size"], stride=layer["stride"])
            # new_maxpool = nn.MaxPool2d(layer["size"])
            network[learner]["maxpool{}".format(lay_cnt)] = new_maxpool

        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer['type'] == "glob_avg_pool":
        in_shape = (input_ch, input_height, input_width)
        out_channels = input_ch
        out_height = 1
        out_width = 1
        out_shape = (out_channels, out_height, out_width)
        
        for learner in range(N_LEARNERS):
            new_glob_avgPool = nn.AdaptiveAvgPool2d((1, 1))
            network[learner]["glob_avgPool{}".format(lay_cnt)] = new_glob_avgPool
        
        input_ch = out_channels
        input_height = out_height
        input_width = out_width

        in_size = out_channels * out_height * out_width

    elif layer["type"] == "dense":
        print("DENSE: ", in_size)
        in_shape = in_size
        out_size = layer["out_size"]

        dense_values, biases_values = generate_template_dense(SAME_SEQ, OUT_FOLDER + "dense_header_{}.h".format(lay_cnt), lay_cnt, N_LEARNERS, CODEBOOK_SIZE, 200, in_size, out_size, USE_F16, USE_CODEBOOKS)
        dense_weights.append(dense_values)

        for learner in range(N_LEARNERS):

            new_dense = nn.Linear(in_shape, layer["out_size"], bias=USE_BIAS)
            
            with torch.no_grad():
                new_dense.weight.copy_(torch.Tensor(dense_values[learner]).view(out_size, in_shape))

                if USE_BIAS:
                    new_dense.bias.copy_(torch.tensor(biases_values[learner]))

            network[learner]["dense{}".format(lay_cnt)] = new_dense

        out_shape = out_size

        in_size = out_shape
        
    else:
        print("ERROR!")
        exit(1)



    print("In shape:", in_shape)
    print("Out shape:", out_shape)
    print()

[0] conv
	 {'type': 'conv', 'in_ch': 3, 'out_ch': 64, 'k_size': 5, 'stride': 1, 'padding': 2}
In shape: (3, 32, 32)
Out shape: (64, 32, 32)

[1] maxpool
	 {'type': 'maxpool', 'size': 3, 'stride': 2}
In shape: (64, 32, 32)
Out shape: (64, 15, 15)

[2] conv
	 {'type': 'conv', 'in_ch': 64, 'out_ch': 192, 'k_size': 3, 'stride': 1, 'padding': 1}


In shape: (64, 15, 15)
Out shape: (192, 15, 15)

[3] maxpool
	 {'type': 'maxpool', 'size': 3, 'stride': 2}
In shape: (192, 15, 15)
Out shape: (192, 7, 7)

[4] conv
	 {'type': 'conv', 'in_ch': 192, 'out_ch': 384, 'k_size': 3, 'stride': 1, 'padding': 1}


In shape: (192, 7, 7)
Out shape: (384, 7, 7)

[5] conv
	 {'type': 'conv', 'in_ch': 384, 'out_ch': 256, 'k_size': 3, 'stride': 1, 'padding': 1}


In shape: (384, 7, 7)
Out shape: (256, 7, 7)

[6] conv
	 {'type': 'conv', 'in_ch': 256, 'out_ch': 256, 'k_size': 3, 'stride': 1, 'padding': 1}


In shape: (256, 7, 7)
Out shape: (256, 7, 7)

[7] glob_avg_pool
	 {'type': 'glob_avg_pool'}
In shape: (256, 7, 7)
Out shape: (256, 1, 1)

[8] dense
	 {'type': 'dense', 'out_size': 4096}
DENSE:  256


In shape: 256
Out shape: 4096

[9] dense
	 {'type': 'dense', 'out_size': 4096}
DENSE:  4096


In shape: 4096
Out shape: 4096

[10] dense
	 {'type': 'dense', 'out_size': 10}
DENSE:  4096


In shape: 4096
Out shape: 10



In [7]:
print(len(biases_values))

for bv in biases_values:
    print(len(bv))
    print(bv)

8
10
[0.57501865 0.8061278  0.89654515 0.89120026 0.16974564 0.36752932
 0.83625318 0.70626013 0.13299824 0.33888708]
10
[0.70287057 0.08428854 0.27054762 0.67821511 0.56070168 0.89196799
 0.28131972 0.61265898 0.29415077 0.21551329]
10
[0.41393547 0.12836644 0.35961204 0.89239863 0.29203195 0.60382687
 0.71054078 0.05694488 0.0024971  0.51569643]
10
[0.85991784 0.5391064  0.31726376 0.75898508 0.11007985 0.53265364
 0.8260462  0.84287638 0.40724639 0.58884339]
10
[0.04741135 0.87103277 0.81138089 0.32663669 0.06728867 0.35644904
 0.51858939 0.02627087 0.17406505 0.03798137]
10
[0.15794465 0.72484139 0.60880425 0.42487777 0.75025425 0.28852875
 0.09499549 0.7196202  0.82474968 0.07063961]
10
[0.16344149 0.57073995 0.27710692 0.54255964 0.77054722 0.16672261
 0.64357477 0.25479087 0.66487077 0.60703072]
10
[0.15492916 0.09864279 0.41355295 0.60412626 0.44969836 0.1254525
 0.50267365 0.85334974 0.69035106 0.80481715]


In [8]:
network[0]['conv0'].bias

Parameter containing:
tensor([0.0940, 0.1063, 0.0605, 0.1362, 0.0239, 0.6835, 0.8472, 0.0598, 0.2443,
        0.1784, 0.8402, 0.0208, 0.5049, 0.2282, 0.0221, 0.0104, 0.6379, 0.5706,
        0.7877, 0.7671, 0.0444, 0.8336, 0.6868, 0.5308, 0.8848, 0.6401, 0.8782,
        0.3889, 0.1705, 0.3686, 0.2508, 0.5465, 0.4387, 0.8977, 0.2622, 0.4701,
        0.5243, 0.1901, 0.3710, 0.3895, 0.6363, 0.1149, 0.7138, 0.6942, 0.7648,
        0.0233, 0.7367, 0.3840, 0.2916, 0.3462, 0.2403, 0.7790, 0.0625, 0.4563,
        0.1442, 0.6866, 0.3645, 0.7287, 0.0512, 0.5106, 0.3437, 0.5861, 0.5864,
        0.5828], requires_grad=True)

In [9]:
input = torch.Tensor(input_values).view(IN_CHANNELS, IN_HEIGHT, IN_WIDTH)

print(input.shape)

for ens in range(N_LEARNERS):
    print("\n=============== LEARNER {} ===============\n".format(ens))
    # print(input)
    x = network[ens]['conv0'](input)
    print(x.shape)
    # print(x)
    # break

    x = torch.relu(x)
    print(x.shape)
    # print(x)
    # break

    x = network[ens]['maxpool1'](x)
    print(x.shape)
    # print(x)
    # break

    x = network[ens]['conv2'](x)
    print(x.shape)
    # print(x)
    # break

    x = torch.relu(x)
    # print(x.shape)
    # print(x)
    # break

    x = network[ens]['maxpool3'](x)
    print(x.shape)
    # print(x)
    # break
        
    x = network[ens]['conv4'](x)
    print(x.shape)
    # print(x)
    # break

    x = torch.relu(x)
        
    x = network[ens]['conv5'](x)
    print(x.shape)
    # print(x)
    # break

    x = torch.relu(x)
        
    x = network[ens]['conv6'](x)
    print(x.shape)
    # print(x)
    # break

    x = torch.relu(x)
    # print(x)
    # break

    # x = network[ens]['maxpool7'](x)
    # print(x.shape)
    # # print(x[0])
    # # break


    x = network[ens]['glob_avgPool7'](x)
    # x = network[ens]['maxpool7'](x)
    print(x.shape)
    # print(x)
    # break

    x = x.flatten()
    print(x.shape)
    # print(x)

    x = network[ens]['dense8'](x)
    print(x.shape)
    # print(x)
    # break

    x = torch.relu(x)
    print(x.shape)
    # print(x[:2])

    x = network[ens]['dense9'](x)
    print(x.shape)
    # print(x)

    x = torch.relu(x)
    # print(x.shape)
    # print(x)

    x = network[ens]['dense10'](x)
    print(x.shape)
    print(x)

    # break

torch.Size([3, 32, 32])

=============== LEARNER 0 ===============

torch.Size([64, 32, 32])
torch.Size([64, 32, 32])
torch.Size([64, 15, 15])
torch.Size([192, 15, 15])
torch.Size([192, 7, 7])
torch.Size([384, 7, 7])
torch.Size([256, 7, 7])
torch.Size([256, 7, 7])
torch.Size([256, 1, 1])
torch.Size([256])
torch.Size([4096])
torch.Size([4096])
torch.Size([4096])
torch.Size([10])
tensor([-1.3078e+15, -1.7249e+15, -1.3943e+15, -1.3731e+15, -1.3224e+15,
        -1.6664e+15, -9.0974e+14, -1.5370e+15, -1.1631e+15, -1.2700e+15],
       grad_fn=<ViewBackward0>)

=============== LEARNER 1 ===============

torch.Size([64, 32, 32])
torch.Size([64, 32, 32])
torch.Size([64, 15, 15])
torch.Size([192, 15, 15])
torch.Size([192, 7, 7])
torch.Size([384, 7, 7])
torch.Size([256, 7, 7])
torch.Size([256, 7, 7])
torch.Size([256, 1, 1])
torch.Size([256])
torch.Size([4096])
torch.Size([4096])
torch.Size([4096])
torch.Size([10])
tensor([-3.0739e+13, -2.1836e+13, -2.9389e+13, -3.0518e+13, -3.0724e+13,
        -2